# FluidX IntelliXcap 96

The FluidX IntelliXcap 96 (an Azenta brand) is an automated screw-cap **decapper**. It unscrews, holds, and screws back on all 96 caps of a screw-cap tube rack in a single stroke. A plate mover loads the rack onto its one nest.

| Property | Value |
|---|---|
| Communication | Serial, STX/ETX-framed ASCII |
| Serial settings | 9600 baud, 8 data bits, no parity, 1 stop bit, no handshake |
| Framing | each reply is wrapped in STX (`0x02`) .. ETX (`0x03`) |
| Reply pattern | ACK (`0x06`), then `<cmd>OK` echo, then a result frame |

Every command is a single character written followed by ETX. A motion command is accepted with an ACK and a `<cmd>OK` echo; the status word then reads `StatusBUSY` while it moves and returns to `StatusOK` when done. A refused or no-op command answers `CommandIgnore`.


```{note}
Connection, status, tray open/close, home, and standby/ready are verified against
hardware. **Decap, recap, and waste engage the cap head and are not yet
hardware-verified** -- validate them on your instrument before relying on them.
`setup()` emits a warning to that effect.
```


## Physical setup

Connect the decapper's serial port to your computer through its USB-to-serial adapter and use the stable `by-id` path so the port survives re-enumeration, e.g. `/dev/serial/by-id/usb-FTDI_USB_Serial_Converter_XXXXXXXX-if00-port0`.

```{important}
If the instrument comes up reporting `StatusBUSY` and ignores commands, it is
locked out. The most common cause is an **engaged e-stop**; the safety guard/hood
and other interlocks do the same. There is no command to read the e-stop
directly, so `setup()` raises with this hint when it sees `StatusBUSY` at connect.
Clear the e-stop and retry.
```


## Connect

`setup()` opens the serial port and reads the status. It raises if the device is not ready (e.g. e-stop), and logs the not-fully-verified warning.


In [ ]:
from pylabrobot.azenta.fluidx import FluidXIntelliXcap96

decapper = FluidXIntelliXcap96(port="/dev/ttyUSB0")  # replace with your port
await decapper.setup()


## Status

`request_status()` returns the current status word: `StatusOK` (idle), `StatusBUSY` (moving), or `StatusSLEEP` (standby). Note the status word does **not** encode the tray position.


In [ ]:
print("status:", await decapper.request_status())


## Tray

Open and close the loading tray. Each call returns once the tray has finished moving (status back to `StatusOK`). Asking for a position the tray is already in is a no-op and raises `FluidXError` (`CommandIgnore`).


In [ ]:
await decapper.open_tray()
await decapper.close_tray()


## Home

Home all axes.


In [ ]:
await decapper.home()


## Decap, recap, waste

With a rack of screw-cap tubes loaded on the nest, `decap()` unscrews and holds all 96 caps, `recap()` screws them back on, and `waste()` drops the held caps into the waste bin. `decap()` refuses if caps are already held (recap first); `recap()` refuses if none are held.

```{warning}
These operations are not yet hardware-verified. Test carefully.
```


In [ ]:
# await decapper.decap()   # unscrew and hold all 96 caps
# await decapper.recap()   # screw the held caps back on
# await decapper.waste()   # drop held caps to the waste bin


## Standby

Put the decapper to sleep with `standby()` and wake it with `ready()`.


In [ ]:
await decapper.standby()
await decapper.ready()


## Teardown

`stop()` closes the serial connection.


In [ ]:
await decapper.stop()
